<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/AIMS2022_Python_Basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Python Basics for Machine Learning

# Table of Contents
1. Installing packages (matminer to load JARVIS 3D dataset)
2. Data manipulation with ``pandas`` package
3. K-means clustering using ``sci-kit learn``
4. PCA dimensionality reduction using ``sci-kit learn``
5. Plotting results using ``matplotlib``

# Python Basics
Python is a highly modular language, which makes the use of external libraries to perform certian operations. We'll first go over the very basics of the pandas and plotly libraries, which will, respectively, be used for data-handling and plotting in the following example.

## Data-handling with Pandas
Pandas is a Python package that defines the DataFrame object. DataFrames are ideal for reading, writing, storing, and manipulating information in a tabular format (i.e. information labelled by a row index and a column heading).


First, we'll fetch the JARVIS-DFT 3D Database, which contains the relaxed structures and DFT properties for about 55,000 compounds using the ``matminer`` Python package. First, we need to install the matminer package using the command below. NOTE: the leading exclamation point (!) indicates that the command should be run as a shell command rather than a Python command.

Uncomment the necessary lines below if your environment is missing any of [matminer](https://hackingmaterials.lbl.gov/matminer/), jarvis-tools or  Scikit-learn.

In [ ]:
!pip install matminer
!pip install -U jarvis-tools
!pip install -U scikit-learn

In [ ]:
import json
import zipfile

import requests
from matminer.datasets.convenience_loaders import load_jarvis_dft_3d

# from jarvis.db.figshare import data
from jarvis.core.atoms import Atoms
from sklearn.cluster import KMeans
from sklearn import preprocessing as pp
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

In [ ]:
# NBVAL_IGNORE_OUTPUT

dft3d_df = load_jarvis_dft_3d(data_home=".")

Properties associated with each entry in the JARVIS-DFT 3D materials database.

In [ ]:
print(dft3d_df.columns)

The `dropna` method drops the rows that contain any NaN (not a number) values.

In [ ]:
dft3d_df = dft3d_df.dropna()

# Fetch the first 50 rows
dft3d_50 = dft3d_df.head(50)

dft3d_50[:10]

Here, we print out all the compositions in our select dataset of 50 materials.

Following function is required due to some issues in Jarvis with downloading data.

In [ ]:
def read_data():
    """Temporary download data function due to
    https://github.com/usnistgov/jarvis/issues/251
    """

    json_tag = "jdft_3d-8-18-2021.json"
    url = "https://ndownloader.figshare.com/files/29204826"
    tmpfile = "tmp.zip"

    response = requests.get(url)
    with open(tmpfile, "wb") as f:
        for block in response.iter_content(1024):
            f.write(block)
    return json.loads(zipfile.ZipFile(tmpfile).read(json_tag))

In [ ]:
dft_3d = read_data()

comp_list = []
for indx, d in dft3d_50.iterrows():
    entry = next(j for j in dft_3d if j["jid"] == d["jid"])
    comp = Atoms.from_dict(entry["atoms"]).composition.reduced_formula
    comp_list.append(comp)

print(comp_list)

Here, we filter out the feature vector to only include single scalar-values properties.

In [ ]:
# Form feature vector from the scalar properties in the dataframe
scalar_cols = [
    col
    for col in dft3d_50.columns
    if col not in ["structure", "structure initial", "jid", "mpid", "composition"]
]

feature_vect = dft3d_50[scalar_cols].to_numpy()

feature_vect[:10]

## Machine Learning Models Implemented in Sci-kit Learn

``sci-kit learn`` is a Python package that faciliates the implementation of several machine learning models for dimensionality reduction, clustering, classification, regression, and more.

Before performing K-means clustering, the data is pre-processed such that each feature has a mean centered at zero and a standard deviation of 1.

**Try modifying**

Run the K--means clustering on the un-scaled feature vector ``feature_vect`` instead to see what happens

The following scales the feature vector as a pre-processing step such that features have zero mean and
unit variance.

In [ ]:
scaler = pp.StandardScaler().fit(feature_vect)

feat_vec_scale = scaler.transform(feature_vect)

print("mean: ", feat_vec_scale.mean(axis=0))
print("std. deviation: ", feat_vec_scale.std(axis=0))

Run the K-means clustering and output the cluster labels for each sample in the dataset.

**Try modifiying**

Change the number of cluster! ``K``

In [ ]:
# NBVAL_IGNORE_OUTPUT
# Stochastic output

# Fit K-means model

K = 5
# CHANGE the number of clusters generated
kmeans = KMeans(n_clusters=K).fit(
    feat_vec_scale
)  # CHANGE to the unscaled/un-whitened feature_vect to see what happens!
lbl = kmeans.labels_
centers = kmeans.cluster_centers_
y_kmeans = kmeans.predict(feat_vec_scale)
print(y_kmeans)

In [ ]:
# NBVAL_IGNORE_OUTPUT
# Stochastic ouput

# Print contents of each cluster

for k in range(K):  # For each generated cluster
    pos = [
        n for (n, m) in enumerate(lbl) if m == k
    ]  # Get the indices in the labels vector which point ot that cluster
    compGrp = np.array(comp_list)[pos]
    print(f"Generated Compound Cluster #{k + 1}")
    print(compGrp)
    print("mean feature values of the cluster:")
    print([{f: m} for f, m in zip(scalar_cols, centers[k])])
    print("\n")

Here, we will use principal component analysis (PCA) primarily as a way to visualize our "high-dimensional" data (11 total features) in two dimensions.

**Try modifying**

Perform the PCA on the feeature vector first and then apply the K-means clustering algorithm to the principal component vectors.

In [ ]:
# Plot samples against first two principal components, colored by their cluster

pca = PCA(n_components=2)
pca.fit(feat_vec_scale)
print(pca)
comp_dim = pca.transform(feat_vec_scale)
plt.figure()
plt.scatter(comp_dim[:, 0], comp_dim[:, 1], c=y_kmeans)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.show()

The principal components are linear combinations of the original features. Here, we plot out how each principal component weights the original features.

In [ ]:
# Show the weights of different features in the first two principal components

pca_weights = pca.components_
fig, (ax1, ax2) = plt.subplots(2, 1)
fig.tight_layout()
ax1.bar(scalar_cols, pca_weights[0])
ax1.set_xticks([])
ax2.bar(scalar_cols, pca_weights[1])
plt.xticks(rotation=30, ha="right")